In [2]:

!pip install -q openenv-core
!git clone --depth=1 -q https://github.com/meta-pytorch/OpenEnv.git 2>/dev/null || true

import sys, os
repo = os.path.abspath('OpenEnv')
for p in [repo, os.path.join(repo, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Setup complete!")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

Setup complete!


The system cannot find the path specified.
'true' is not recognized as an internal or external command,
operable program or batch file.


In [10]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

# These would normally go in models.py

@dataclass
class EmailTriageAction:
    """
    Agent performs an action depending on stage:
    - classification → spam / important / support
    - intent → pricing / complaint / booking / etc
    - reply → generated text
    """
    action_type: str   # "classification" | "intent" | "reply"
    content: str       
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class EmailTriageObservation:
    """
    What the agent sees after each step
    """
    done: bool
    reward: Optional[float]

    email_text: str
    current_stage: str   # classification → intent → reply
    history: List[Dict[str, str]]  # past actions

    message: str   # feedback
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class EmailTriageState:
    """
    Internal environment state
    """
    episode_id: Optional[str] = None
    step_count: int = 0

    email_text: str = ""
    true_classification: str = ""
    true_intent: str = ""
    true_reply: str = ""

    current_stage: str = "classification"


print("Types defined: EmailTriageAction, EmailTriageObservation, EmailTriageState")

Types defined: EmailTriageAction, EmailTriageObservation, EmailTriageState


In [15]:
import random
import uuid

# Sample dataset (you will expand later)
EMAILS = [
    {
        "email": "Hi, I want to know the price of solar panels for my home.",
        "classification": "important",
        "intent": "pricing inquiry",
        "reply": "Thank you for your interest. We will share pricing details shortly."
    },
    {
        "email": "My solar system is not working properly. Need urgent support.",
        "classification": "support",
        "intent": "complaint",
        "reply": "We’re sorry for the inconvenience. Our support team will contact you shortly."
    },
    {
        "email": "Can you schedule a visit for installation next week?",
        "classification": "important",
        "intent": "booking",
        "reply": "Sure, we will arrange a visit and confirm the schedule shortly."
    },
    {
        "email": "What is the maintenance cost for solar panels?",
        "classification": "important",
        "intent": "pricing inquiry",
        "reply": "We will provide details about maintenance costs soon."
    },
    {
        "email": "I am facing issues with my inverter. Please help.",
        "classification": "support",
        "intent": "complaint",
        "reply": "Our support team will reach out to resolve the issue."
    },
    {
        "email": "Do you offer EMI options for solar installation?",
        "classification": "important",
        "intent": "pricing inquiry",
        "reply": "Yes, we offer EMI options. Our team will share details."
    },
    {
        "email": "Book a demo for solar installation at my house.",
        "classification": "important",
        "intent": "booking",
        "reply": "We will schedule a demo and confirm shortly."
    },
    {
        "email": "System stopped working after last update.",
        "classification": "support",
        "intent": "complaint",
        "reply": "We’re sorry for the issue. Our team will assist you soon."
    },
    {
        "email": "Limited time offer! Get free solar panels now!",
        "classification": "spam",
        "intent": "promotion",
        "reply": "This appears to be spam. No action required."
    },
    {
        "email": "Congratulations! You won a free gift card. Click here.",
        "classification": "spam",
        "intent": "promotion",
        "reply": "This is likely spam. Avoid clicking suspicious links."
    },
    {
        "email": "Is there any subsidy available for solar panels?",
        "classification": "important",
        "intent": "pricing inquiry",
        "reply": "We will share information about subsidies shortly."
    },
    {
        "email": "Need help with installation. It is delayed.",
        "classification": "support",
        "intent": "complaint",
        "reply": "We apologize for the delay. Our team will assist you."
    }
]


class EmailTriageEnvironment:
    """Email triage environment following OpenEnv pattern."""

    def __init__(self):
        self._state = EmailTriageState()
        self._current_email = None
        self._history = []

    def reset(self) -> EmailTriageObservation:
        """Start a new episode with a random email."""
        self._current_email = random.choice(EMAILS)
        self._history = []

        self._state = EmailTriageState(
            episode_id=str(uuid.uuid4()),
            step_count=0,
            email_text=self._current_email["email"],
            true_classification=self._current_email["classification"],
            true_intent=self._current_email["intent"],
            true_reply=self._current_email["reply"],
            current_stage="classification"
        )

        return EmailTriageObservation(
            done=False,
            reward=None,
            email_text=self._state.email_text,
            current_stage="classification",
            history=[],
            message="Start with classification"
        )

    def step(self, action: EmailTriageAction) -> EmailTriageObservation:
        """Process agent action."""
        self._state.step_count += 1
        stage = self._state.current_stage
        reward = 0.0
        message = ""

        # --- Classification Stage ---
        if stage == "classification":
            if action.content == self._state.true_classification:
                reward = 0.3
                message = "Correct classification"
            else:
                message = "Wrong classification"

            self._history.append({"stage": stage, "output": action.content})
            self._state.current_stage = "intent"

        # --- Intent Stage ---
        elif stage == "intent":
            if action.content == self._state.true_intent:
                reward = 0.3
                message = "Correct intent"
            else:
                message = "Wrong intent"

            self._history.append({"stage": stage, "output": action.content})
            self._state.current_stage = "reply"

        # --- Reply Stage ---
        elif stage == "reply":
            if action.content.lower() in self._state.true_reply.lower():
                reward = 0.4
                message = "Good reply"
            else:
                reward = 0.2  # partial credit
                message = "Acceptable reply"

            self._history.append({"stage": stage, "output": action.content})

            return EmailTriageObservation(
                done=True,
                reward=reward,
                email_text=self._state.email_text,
                current_stage="done",
                history=self._history,
                message=message
            )

        return EmailTriageObservation(
            done=False,
            reward=reward,
            email_text=self._state.email_text,
            current_stage=self._state.current_stage,
            history=self._history,
            message=message
        )

    @property
    def state(self) -> EmailTriageState:
        return self._state


print("EmailTriageEnvironment defined.")

EmailTriageEnvironment defined.


In [16]:
env = EmailTriageEnvironment()
obs = env.reset()

print(f"Email: {obs.email_text}")
print(f"Stage: {obs.current_stage}")
print(f"Message: {obs.message}")
print()

# Step 1: Classification
obs = env.step(EmailTriageAction(action_type="classification", content="important"))
print(f"[Classification] → Stage: {obs.current_stage}, Reward: {obs.reward}, Message: {obs.message}")

# Step 2: Intent
obs = env.step(EmailTriageAction(action_type="intent", content="pricing inquiry"))
print(f"[Intent] → Stage: {obs.current_stage}, Reward: {obs.reward}, Message: {obs.message}")

# Step 3: Reply
obs = env.step(EmailTriageAction(action_type="reply", content="We will share pricing details soon."))
print(f"[Reply] → Stage: {obs.current_stage}, Reward: {obs.reward}, Message: {obs.message}")

print("\nFinal:")
print(f"Done: {obs.done}")
print(f"Total Steps: {env.state.step_count}")
print(f"Episode ID: {env.state.episode_id[:8]}...")
print(f"History: {obs.history}")

Email: What is the maintenance cost for solar panels?
Stage: classification
Message: Start with classification

[Classification] → Stage: intent, Reward: 0.3, Message: Correct classification
[Intent] → Stage: reply, Reward: 0.3, Message: Correct intent
[Reply] → Stage: done, Reward: 0.2, Message: Acceptable reply

Final:
Done: True
Total Steps: 3
Episode ID: c22f6d22...
History: [{'stage': 'classification', 'output': 'important'}, {'stage': 'intent', 'output': 'pricing inquiry'}, {'stage': 'reply', 'output': 'We will share pricing details soon.'}]


In [19]:
import random

class RandomEmailPolicy:
    """Random decisions for all stages"""
    name = "Random Policy"

    classifications = ["spam", "important", "support"]
    intents = ["pricing inquiry", "complaint", "booking", "general question"]

    def select_action(self, obs: EmailTriageObservation) -> EmailTriageAction:
        stage = obs.current_stage

        if stage == "classification":
            return EmailTriageAction(
                action_type="classification",
                content=random.choice(self.classifications)
            )

        elif stage == "intent":
            return EmailTriageAction(
                action_type="intent",
                content=random.choice(self.intents)
            )

        elif stage == "reply":
            return EmailTriageAction(
                action_type="reply",
                content="Thank you for reaching out. We will get back to you soon."
            )
        


class RuleBasedEmailPolicy:
    name = "Rule-Based Policy"

    def select_action(self, obs: EmailTriageObservation) -> EmailTriageAction:
        email = obs.email_text.lower()
        stage = obs.current_stage

        # --- Classification ---
        if stage == "classification":
            if any(word in email for word in ["free", "click", "offer", "win"]):
                return EmailTriageAction("classification", "spam")
            elif any(word in email for word in ["not working", "issue", "problem", "help", "delay"]):
                return EmailTriageAction("classification", "support")
            else:
                return EmailTriageAction("classification", "important")

        # --- Intent ---
        elif stage == "intent":
            if any(word in email for word in ["price", "cost", "emi", "subsidy"]):
                return EmailTriageAction("intent", "pricing inquiry")
            elif any(word in email for word in ["not working", "problem", "issue", "delay"]):
                return EmailTriageAction("intent", "complaint")
            elif any(word in email for word in ["book", "schedule", "demo", "visit"]):
                return EmailTriageAction("intent", "booking")
            else:
                return EmailTriageAction("intent", "general question")

        # --- Reply ---
        elif stage == "reply":
            if any(word in email for word in ["price", "cost", "emi", "subsidy"]):
                return EmailTriageAction(
                    "reply",
                    "We will share pricing and subsidy details shortly."
                )
            elif any(word in email for word in ["not working", "issue", "problem"]):
                return EmailTriageAction(
                    "reply",
                    "We’re sorry for the issue. Our support team will contact you soon."
                )
            elif any(word in email for word in ["book", "schedule", "demo"]):
                return EmailTriageAction(
                    "reply",
                    "We will schedule your request and confirm shortly."
                )
            else:
                return EmailTriageAction(
                    "reply",
                    "Thank you for reaching out. We will get back to you soon."
                )


def evaluate(env, policy, episodes=20):
    
    total_score = 0
    total_steps = 0

    for _ in range(episodes):
        obs = env.reset()
        episode_score = 0

        while not obs.done:
            action = policy.select_action(obs)
            obs = env.step(action)

            if obs.reward is not None:
                episode_score += obs.reward   # ✅ accumulate

        total_score += episode_score
        total_steps += env.state.step_count

    avg_score = total_score / episodes
    avg_steps = total_steps / episodes

    return avg_score, avg_steps




env = EmailTriageEnvironment()

for policy in [RandomEmailPolicy(), RuleBasedEmailPolicy()]:
    score, steps = evaluate(env, policy)
    print(f"{policy.name:20s} → Avg Score: {score:.2f}, Avg Steps: {steps}")

Random Policy        → Avg Score: 0.37, Avg Steps: 3.0
Rule-Based Policy    → Avg Score: 0.73, Avg Steps: 3.0


In [20]:
def rollout(env, policy):
    obs = env.reset()

    print("[START]")
    print(f"Email: {obs.email_text}")
    print(f"Stage: {obs.current_stage}")
    print()

    total_reward = 0

    while not obs.done:
        action = policy.select_action(obs)

        print("[STEP]")
        print(f"Stage: {obs.current_stage}")
        print(f"Action: {action.content}")

        obs = env.step(action)

        print(f"Reward: {obs.reward}")
        print(f"Next Stage: {obs.current_stage}")
        print(f"Message: {obs.message}")
        print()

        if obs.reward:
            total_reward += obs.reward

    print("[END]")
    print(f"Final Score: {total_reward}")
    print(f"Steps: {env.state.step_count}")

env = EmailTriageEnvironment()
policy = RuleBasedEmailPolicy()

rollout(env, policy)

[START]
Email: Hi, I want to know the price of solar panels for my home.
Stage: classification

[STEP]
Stage: classification
Action: important
Reward: 0.3
Next Stage: intent
Message: Correct classification

[STEP]
Stage: intent
Action: pricing inquiry
Reward: 0.3
Next Stage: reply
Message: Correct intent

[STEP]
Stage: reply
Action: We will share pricing and subsidy details shortly.
Reward: 0.2
Next Stage: done
Message: Acceptable reply

[END]
Final Score: 0.8
Steps: 3
